# Exam Scheduling — Experiment Notebook

8 algorithms (7 C++ + IP solver). Falls back to Python automatically if C++ isn't compiled.

**Workflow:** Edit config &rarr; Run experiments &rarr; View tables & plots &rarr; Export

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import importlib
import pandas as pd
from IPython.display import display, HTML

from data.parser import parse_itc2007_exam
from data.generator import generate_synthetic, write_itc2007_format
from algorithms.cpp_bridge import run_solver
from utils.results_logger import ResultsLogger

# Force-reload plotting so re-running this cell picks up any changes
import utils.plotting
importlib.reload(utils.plotting)
from utils.plotting import *
import matplotlib.pyplot as plt

try:
    from algorithms.ip_solver import solve_ip
    HAS_IP = True
except ImportError:
    HAS_IP = False

logger = ResultsLogger("results/run_log.jsonl")
print(f"Existing records: {len(logger.load_all())}")
print("Ready. All solvers route through C++ binary.")

## Config

In [ ]:
DATASETS = {
    "exam_comp_set1": "datasets/exam_comp_set1.exam",
    "exam_comp_set2": "datasets/exam_comp_set2.exam",
    "exam_comp_set3": "datasets/exam_comp_set3.exam",
    "exam_comp_set4": "datasets/exam_comp_set4.exam",
    "exam_comp_set5": "datasets/exam_comp_set5.exam",
    "exam_comp_set6": "datasets/exam_comp_set6.exam",
    "exam_comp_set7": "datasets/exam_comp_set7.exam",
    "exam_comp_set8": "datasets/exam_comp_set8.exam"
}

# Algorithm to run: "greedy", "tabu", "hho", "kempe", "sa", "alns", "gd", or "all"
ALGO = "all"

# Algorithm parameters
TABU_ITERS     = 1000
TABU_TENURE    = 20
TABU_PATIENCE  = 500
HHO_POP        = 30
HHO_ITERS      = 150
SA_ITERS       = 3000
KEMPE_ITERS    = 2000
ALNS_ITERS     = 1000
GD_ITERS       = 3000
SEED           = 42
NUM_TRIALS     = 10

# IP solver (Python, only for small instances)
RUN_IP         = False
IP_TIME_LIMIT  = 600

VERBOSE = True

## Load Datasets

In [ ]:
problems = {}
for name, path in DATASETS.items():
    if os.path.isfile(path):
        problems[name] = parse_itc2007_exam(path)
        p = problems[name]
        print(f"{name}: {p.num_exams()} exams, {p.num_periods()} periods, "
              f"{p.num_rooms()} rooms, {p.num_students()} students, "
              f"density={p.conflict_density():.3f}")
    else:
        print(f"SKIP: {path} not found")
print(f"\nLoaded {len(problems)} dataset(s)")

## Run Experiments

In [ ]:
session_records = []

for ds_name, ds_path in DATASETS.items():
    if ds_name not in problems:
        continue
    problem = problems[ds_name]
    print(f"\n{'='*60}\n  {ds_name} ({problem.num_exams()} exams)\n{'='*60}")

    for trial in range(NUM_TRIALS):
        seed = SEED + trial * 1000

        results = run_solver(
            ds_path, algo=ALGO,
            tabu_iters=TABU_ITERS, tabu_tenure=TABU_TENURE,
            tabu_patience=TABU_PATIENCE, hho_pop=HHO_POP,
            hho_iters=HHO_ITERS, sa_iters=SA_ITERS,
            kempe_iters=KEMPE_ITERS, alns_iters=ALNS_ITERS,
            gd_iters=GD_ITERS, seed=seed, verbose=VERBOSE,
        )

        for algo_name, r in results.items():
            config = {
                "tabu_iters": TABU_ITERS, "tabu_tenure": TABU_TENURE,
                "hho_pop": HHO_POP, "hho_iters": HHO_ITERS,
                "sa_iters": SA_ITERS, "kempe_iters": KEMPE_ITERS,
                "alns_iters": ALNS_ITERS, "gd_iters": GD_ITERS,
                "seed": seed,
            }
            rec = logger.log_run(ds_name, problem, r, config=config, trial=trial)
            session_records.append(rec)
            ev = r['evaluation']
            feasible = ev.feasible if hasattr(ev, 'feasible') else ev.is_feasible
            soft = ev.soft if hasattr(ev, 'soft') else ev.soft_penalty
            hard = ev.hard if hasattr(ev, 'hard') else ev.hard_violations
            print(f"  [{algo_name:<22}] t={trial}  feasible={feasible}  "
                  f"hard={hard}  soft={soft:>8}  time={r['runtime']:.3f}s")

        # IP solver (Python, optional)
        if RUN_IP and HAS_IP and problem.num_exams() <= 300:
            r = solve_ip(problem, time_limit=IP_TIME_LIMIT, verbose=VERBOSE)
            rec = logger.log_run(ds_name, problem, r,
                config={"ip_time_limit": IP_TIME_LIMIT}, trial=trial)
            session_records.append(rec)

print(f"\nLogged {len(session_records)} records this session")
print(f"Total in log: {len(logger.load_all())}")

## Results Table

In [ ]:
df = logger.to_dataframe()

summary = df.groupby(['dataset', 'algorithm']).agg(
    runs=('feasible', 'count'),
    feasible_pct=('feasible', lambda x: f"{x.mean()*100:.0f}%"),
    hard_mean=('hard_violations', 'mean'),
    soft_mean=('soft_penalty', 'mean'),
    soft_std=('soft_penalty', 'std'),
    soft_min=('soft_penalty', 'min'),
    runtime_mean=('runtime', 'mean'),
).round(2).reset_index()

display(HTML("<h3>All Logged Runs</h3>"))
display(summary)


## Aggregated Results & Export

In [ ]:
agg_path = logger.save_aggregated()
csv_path = logger.to_csv()
print(f"Aggregated JSON: {agg_path}")
print(f"Full CSV: {csv_path}")

agg_df = logger.aggregate_to_dataframe()
cols = ['algorithm','dataset','count','feasible_rate',
        'runtime_mean','runtime_std','soft_penalty_mean','soft_penalty_std','soft_penalty_min']
display(agg_df[[c for c in cols if c in agg_df.columns]].round(2))

## Visualizations

Charts are horizontal-bar + line where possible to avoid label overlap.  
Radar, box, and ranking plots provide the multi-dimensional view expected in research papers.

In [ ]:
# ── Soft penalty + runtime comparison (horizontal bars) ──
for ds in df['dataset'].unique():
    fig, axes = plt.subplots(1, 2, figsize=(18, max(3.5, len(df['algorithm'].unique()) * 0.7)))
    plot_algorithm_comparison(df, dataset=ds, metric='soft_penalty', ax=axes[0])
    plot_algorithm_comparison(df, dataset=ds, metric='runtime', ax=axes[1])
    fig.suptitle(ds, fontsize=14, fontweight='bold')
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

In [ ]:
# ── Soft constraint breakdown (stacked bar + component lines) ──
for ds in df['dataset'].unique():
    plot_soft_breakdown(df, dataset=ds)
    plt.show()

    plot_soft_lines(df, dataset=ds)
    plt.show()

In [ ]:
# ── Box plot — distribution across runs ──
for ds in df['dataset'].unique():
    plot_box_comparison(df, dataset=ds, metric='soft_penalty')
    plt.show()

In [ ]:
# ── Radar — normalized performance profile (smaller = better) ──
for ds in df['dataset'].unique():
    plot_radar(df, dataset=ds)
    plt.show()

In [ ]:
# ── Quality vs runtime scatter ──
for ds in df['dataset'].unique():
    plot_runtime_vs_quality(df, dataset=ds)
    plt.show()

In [ ]:
# ── Ranking table ──
for ds in df['dataset'].unique():
    plot_rank_table(df, dataset=ds)
    plt.show()

In [ ]:
# ── Cross-dataset: heatmap + line comparison + feasibility ──
# (only useful with 2+ datasets)
if len(df['dataset'].unique()) > 1:
    plot_multi_dataset_heatmap(df, metric='soft_penalty')
    plt.show()

    plot_line_across_datasets(df, metric='soft_penalty')
    plt.show()

    plot_line_across_datasets(df, metric='runtime',
                             title='Runtime Across Datasets')
    plt.show()

    plot_feasibility_rates(df)
    plt.show()
else:
    plot_feasibility_rates(df)
    plt.show()

In [ ]:
# ── Full dashboard (2x2 summary) ──
for ds in df['dataset'].unique():
    plot_summary_dashboard(df, dataset=ds)
    plt.show()

In [ ]:
# ── Save all plots to results/ ──
generate_all_plots(logger, output_dir='results')

## Parameter Sweep

Runs a single algorithm with increasing iteration counts to study convergence.
Change `SWEEP_ALGO` to sweep any algorithm.

In [ ]:
"""
SWEEP_ALGO    = "tabu"          # "tabu", "sa", "kempe", "alns", "gd", "hho"
SWEEP_PARAM   = "tabu_iters"    # kwarg name passed to run_solver
SWEEP_DATASET = list(DATASETS.keys())[0]
SWEEP_PATH    = DATASETS[SWEEP_DATASET]
PARAM_VALUES  = [100, 250, 500, 1000, 2000, 3000]
SWEEP_TRIALS  = 3

sweep_records = []
problem = problems[SWEEP_DATASET]

for val in PARAM_VALUES:
    for trial in range(SWEEP_TRIALS):
        seed = 42 + trial * 1000
        results = run_solver(
            SWEEP_PATH, algo=SWEEP_ALGO,
            seed=seed, verbose=False,
            **{SWEEP_PARAM: val},
        )
        for name, r in results.items():
            rec = logger.log_run(SWEEP_DATASET, problem, r,
                config={SWEEP_PARAM: val, "seed": seed},
                trial=trial, notes=f"sweep_{SWEEP_PARAM}_{val}")
            sweep_records.append(rec)
        print(f"  {SWEEP_PARAM}={val} t={trial}: soft={rec['soft_penalty']} time={rec['runtime']:.2f}s")

sdf = pd.DataFrame(sweep_records)
sdf['sweep_val'] = sdf['config'].apply(lambda c: c.get(SWEEP_PARAM, 0))
plot_parameter_sensitivity(sdf, 'sweep_val', metric='soft_penalty', algorithm=sweep_records[0]['algorithm'])
plt.xlabel(SWEEP_PARAM)
plt.show()
"""

## Scaling Study — Synthetic Instances

Generates competition-level instances of increasing size. Uses presets:
`"easy"`, `"medium"`, `"hard"`, `"competition"`.


In [ ]:
"""
SIZES  = [1000]
PRESET = "hard"     # easy, medium, hard, competition

os.makedirs("datasets", exist_ok=True)
scale_records = []

for n in SIZES:
    problem = generate_synthetic(num_exams=n, preset=PRESET, seed=42+n)
    exam_path = f"datasets/synthetic_{PRESET}_{n}.exam"
    write_itc2007_format(problem, exam_path)

    for trial in range(2):
        seed = 42 + trial * 1000
        results = run_solver(exam_path, algo="all",
            tabu_iters=min(2000, n*5), hho_iters=min(200, n),
            hho_pop=min(30, max(10, n//10)),
            seed=seed, verbose=False)

        for name, r in results.items():
            rec = logger.log_run(f"synthetic_{n}", problem, r,
                config={"preset": PRESET, "seed": seed},
                trial=trial, notes=f"scaling_{PRESET}")
            scale_records.append(rec)

    print(f"  n={n}: {len(results)} algos done")

# Line charts for scaling (line > bar for continuous x-axis)
sdf = pd.DataFrame(scale_records)
if len(SIZES) > 1:
    plot_scaling(sdf, x_col='num_exams', y_col='runtime')
    plt.show()
    plot_scaling(sdf, x_col='num_exams', y_col='soft_penalty')
    plt.show()
else:
    # Single size — show bar comparison instead
    plot_algorithm_comparison(sdf, metric='soft_penalty',
                             title=f'Soft Penalty — synthetic_{SIZES[0]}')
    plt.show()
    plot_algorithm_comparison(sdf, metric='runtime',
                             title=f'Runtime — synthetic_{SIZES[0]}')
    plt.show()
"""

## Custom Experiment

In [ ]:
# Uncomment and edit as needed
#
# results = run_solver("datasets/exam_comp_set4.exam", algo="sa",
#                       sa_iters=10000, seed=123, verbose=True)
#
# p = problems["exam_comp_set4"]
# for name, r in results.items():
#     rec = logger.log_run("exam_comp_set4", p, r,
#         config={"sa_iters": 10000}, notes="custom experiment")
#     print(f"{name}: soft={rec['soft_penalty']} time={rec['runtime']:.2f}s")

## Reset & Utilities

**Log stats** — shows how many records and file size.  
**Quick Reset** — clears all logged results so you can start fresh.

In [ ]:
# ─── Log stats ───
if os.path.isfile(logger.filepath):
    n = len(logger.load_all())
    kb = os.path.getsize(logger.filepath) / 1024
    print(f"Log: {n} records, {kb:.1f} KB")
else:
    print("No log file found.")

In [ ]:
# === QUICK RESET — run this cell to clear everything ===
"""
logger.clear()
session_records.clear()
if 'df' in dir(): del df
print("All results cleared. Re-run from the top.")
"""